<a href="https://colab.research.google.com/github/TVSRIDURGA/ARES/blob/main/Project_Ares.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install the engine and the tunnel
!pip install streamlit pyngrok opencv-python-headless torch segmentation-models-pytorch scikit-image -q

# Authenticate ngrok
import os
os.system("ngrok config add-authtoken 3BIsHcBq4hrafFQwQUnMkda1zN8_41zk6LznS2U4CTcnQd4wT")
print("Environment Ready!")

Environment Ready!


In [ ]:
%%writefile app.py
import streamlit as st
import cv2
import torch
import numpy as np
import segmentation_models_pytorch as smp
from skimage.metrics import structural_similarity as ssim
import io
from PIL import Image as PILImage

st.set_page_config(page_title="A.R.E.S.", layout="wide", initial_sidebar_state="collapsed")

st.markdown("""
    <style>
    @import url('https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@300;400;700&family=Inter:wght@300;400;600;700;900&display=swap');

    /* ══════════════════════════════════════
       60% — CHARCOAL #1C1C1E  (base/background)
       30% — DEEP MARINE #0A1628 (cards/surfaces)
       10% — BRIGHT MARINE #00A8E8 (accent/glow)
    ══════════════════════════════════════ */

    /* ── BASE (60%) ── */
    .stApp {
        background-color: #1C1C1E;
        color: #E8EDF2;
        font-family: 'Inter', sans-serif;
    }
    .block-container { padding: 2rem 3rem 4rem 3rem !important; max-width: 1400px; }
    #MainMenu, footer, header { visibility: hidden; }
    .stSpinner > div { border-top-color: #00A8E8 !important; }
    ::-webkit-scrollbar { width: 4px; }
    ::-webkit-scrollbar-track { background: #1C1C1E; }
    ::-webkit-scrollbar-thumb { background: #0A1628; border-radius: 2px; }
    ::-webkit-scrollbar-thumb:hover { background: #00A8E8; }

    /* ── SCANLINES ── */
    .stApp::before {
        content: '';
        position: fixed;
        top: 0; left: 0;
        width: 100%; height: 100%;
        background: repeating-linear-gradient(
            0deg, transparent, transparent 2px,
            rgba(0, 168, 232, 0.008) 2px,
            rgba(0, 168, 232, 0.008) 4px
        );
        pointer-events: none;
        z-index: 0;
    }

    /* ── HERO ── */
    .hero {
        padding: 60px 0 40px 0;
        border-bottom: none;
        margin-bottom: 0px;
    }
    .hero-eyebrow {
        font-family: 'JetBrains Mono', monospace;
        font-size: 0.7rem;
        color: #00A8E8;
        letter-spacing: 5px;
        text-transform: uppercase;
        margin-bottom: 16px;
        opacity: 0.8;
    }
    .hero-title {
        font-size: 7rem;
        font-weight: 900;
        letter-spacing: -4px;
        line-height: 0.85;
        color: #FFFFFF;
        margin: 0;
        text-shadow: 0 0 120px rgba(0,168,232,0.12);
    }
    .hero-title .dot {
        color: #00A8E8;
        text-shadow: 0 0 30px rgba(0,168,232,0.9), 0 0 60px rgba(0,168,232,0.4);
    }
    .hero-fullname {
        font-family: 'JetBrains Mono', monospace;
        font-size: 0.68rem;
        color: #00A8E8;
        letter-spacing: 6px;
        margin-top: 18px;
        opacity: 0.55;
    }
    .hero-sub {
        font-family: 'JetBrains Mono', monospace;
        font-size: 0.65rem;
        color: #2A3A4A;
        letter-spacing: 2px;
        margin-top: 10px;
    }

    /* ── DIVIDER (10% accent) ── */
    .marine-line {
        width: 100%;
        height: 1px;
        background: linear-gradient(90deg, transparent, #00A8E855, transparent);
        margin: 20px 0;
    }

    /* ── PIPELINE (30% surface) ── */
    .pipeline {
        display: flex;
        align-items: center;
        flex-wrap: wrap;
        gap: 4px;
        margin: 8px 0 16px 0;
    }
    .pipe-step {
        font-family: 'JetBrains Mono', monospace;
        font-size: 0.58rem;
        color: #00A8E8;
        letter-spacing: 1px;
        padding: 5px 11px;
        border: 1px solid #0A2035;
        border-radius: 2px;
        background: #0A1628;
        text-transform: uppercase;
        opacity: 0.85;
        cursor: default;
        transition: all 0.25s ease;
    }
    .pipe-step:hover {
        opacity: 1;
        border-color: #00A8E8;
        background: #00A8E812;
        box-shadow: 0 0 12px rgba(0,168,232,0.2), inset 0 0 8px rgba(0,168,232,0.05);
        text-shadow: 0 0 8px rgba(0,168,232,0.6);
        transform: translateY(-1px);
    }
    .pipe-arrow { font-size: 0.55rem; color: #0A2035; padding: 0 2px; }

    /* ── UPLOAD (30% surface) ── */
    .upload-label {
        font-family: 'JetBrains Mono', monospace;
        font-size: 0.65rem;
        color: #00A8E8;
        letter-spacing: 3px;
        margin-bottom: 8px;
        opacity: 0.7;
    }
    .stFileUploader label { display: none !important; }
    .stFileUploader section {
        background: #0A1628 !important;
        border: 1px dashed #00A8E844 !important;
        border-radius: 4px !important;
        padding: 36px !important;
        transition: all 0.3s ease !important;
    }
    .stFileUploader section:hover {
        border-color: #00A8E8 !important;
        box-shadow: 0 0 24px rgba(0,168,232,0.08) !important;
    }
    .stFileUploader section p {
        color: #00A8E8AA !important;
        font-family: 'JetBrains Mono', monospace !important;
        font-size: 0.68rem !important;
        letter-spacing: 2px !important;
    }
    .stFileUploader section small {
        color: #00A8E844 !important;
        font-family: 'JetBrains Mono', monospace !important;
    }
    [data-testid="stFileUploaderDropzoneInstructions"] div span {
        color: #00A8E8AA !important;
        font-family: 'JetBrains Mono', monospace !important;
        font-size: 0.75rem !important;
        letter-spacing: 2px !important;
    }
    [data-testid="stFileUploaderDropzoneInstructions"] div small {
        color: #00A8E844 !important;
        font-family: 'JetBrains Mono', monospace !important;
        font-size: 0.6rem !important;
        letter-spacing: 1px !important;
    }
    .stFileUploader svg { fill: #00A8E855 !important; }
    .stFileUploader [data-testid="stBaseButton-secondary"] {
        background: transparent !important;
        border: 1px solid #00A8E844 !important;
        color: #00A8E8 !important;
        font-family: 'JetBrains Mono', monospace !important;
        font-size: 0.62rem !important;
        letter-spacing: 2px !important;
        border-radius: 2px !important;
        transition: all 0.3s !important;
    }
    .stFileUploader [data-testid="stBaseButton-secondary"]:hover {
        border-color: #00A8E8 !important;
        background: #00A8E811 !important;
        box-shadow: 0 0 12px rgba(0,168,232,0.15) !important;
    }

    /* ── IMAGE CARDS (30% surface) ── */
    .img-card {
        background: #0A1628;
        border: 1px solid #0F2035;
        border-radius: 4px;
        padding: 24px;
        transition: border 0.3s ease;
    }
    .img-card:hover { border-color: #00A8E822; }
    .img-card-enhanced {
        background: #0A1628;
        border: 1px solid #00A8E833;
        border-radius: 4px;
        padding: 24px;
        box-shadow: 0 0 40px rgba(0,168,232,0.07);
    }
    .img-card-enhanced:hover { border-color: #00A8E866; }
    .img-label {
        font-family: 'JetBrains Mono', monospace;
        font-size: 0.62rem;
        letter-spacing: 3px;
        text-transform: uppercase;
        margin-bottom: 16px;
        display: flex;
        align-items: center;
        gap: 8px;
    }
    .img-label-orig { color: #3A4A5A; }
    .img-label-enh  { color: #00A8E8; }
    .img-label-dot  { width: 6px; height: 6px; border-radius: 50%; display: inline-block; }
    .dot-gray   { background: #3A4A5A; }
    .dot-marine { background: #00A8E8; box-shadow: 0 0 8px #00A8E8, 0 0 16px rgba(0,168,232,0.5); }

    /* ── METRICS (30% surface, 10% accent) ── */
    .metric-box {
        background: #0A1628;
        border: 1px solid #0F2035;
        border-radius: 4px;
        padding: 22px 20px;
        transition: border 0.3s, box-shadow 0.3s;
    }
    .metric-box:hover {
        border-color: #00A8E844;
        box-shadow: 0 0 20px rgba(0,168,232,0.06);
    }
    .m-label {
        font-family: 'JetBrains Mono', monospace;
        font-size: 0.55rem;
        color: #1A3A5A;
        letter-spacing: 3px;
        text-transform: uppercase;
        margin-bottom: 8px;
    }
    .m-value {
        font-family: 'JetBrains Mono', monospace;
        font-size: 1.9rem;
        font-weight: 700;
        color: #00A8E8;
        text-shadow: 0 0 24px rgba(0,168,232,0.5);
        letter-spacing: -1px;
        line-height: 1.1;
        margin-bottom: 10px;
    }
    .m-delta {
        font-family: 'JetBrains Mono', monospace;
        font-size: 0.55rem;
        color: #00A8E888;
        letter-spacing: 1px;
        background: #00A8E80A;
        border: 1px solid #00A8E822;
        border-radius: 2px;
        padding: 3px 8px;
        display: inline-block;
    }

    /* ── DOWNLOAD BUTTON (10% accent) ── */
    .stDownloadButton button {
        background: transparent !important;
        border: 1px solid #00A8E844 !important;
        color: #00A8E8 !important;
        font-family: 'JetBrains Mono', monospace !important;
        font-size: 0.65rem !important;
        letter-spacing: 3px !important;
        padding: 14px 24px !important;
        border-radius: 2px !important;
        transition: all 0.3s ease !important;
        text-transform: uppercase !important;
        width: 100% !important;
    }
    .stDownloadButton button:hover {
        background: #00A8E811 !important;
        border-color: #00A8E8 !important;
        box-shadow: 0 0 24px rgba(0,168,232,0.15) !important;
    }

    /* ── STATUS BAR ── */
    .status-bar {
        font-family: 'JetBrains Mono', monospace;
        font-size: 0.55rem;
        color: #1A3A5A;
        padding: 14px 0 8px 0;
        border-top: 1px solid #0F2035;
        margin-top: 48px;
        display: flex;
        justify-content: space-between;
        letter-spacing: 2px;
    }

    /* ── EMPTY STATE ── */
    .empty-state {
        border: 1px dashed #0F2035;
        border-radius: 4px;
        padding: 100px 40px;
        text-align: center;
        background: #0A1628;
        margin-top: 16px;
    }
    </style>
""", unsafe_allow_html=True)

# --- ENGINE (UNTOUCHED) ---
@st.cache_resource
def load_engine():
    m = smp.Unet('resnet18', encoder_weights='imagenet', classes=1, activation='sigmoid')
    m.eval()
    return m

def process_reva(orig, model):
    low = cv2.resize(orig, (224, 224))
    rgb = cv2.cvtColor(low, cv2.COLOR_GRAY2RGB)
    tensor = torch.from_numpy(rgb).permute(2, 0, 1).unsqueeze(0).float() / 255.0
    with torch.no_grad():
        mask = model(tensor).squeeze().numpy()
    denoised = cv2.medianBlur(orig, 3)
    _, m_art = cv2.threshold(denoised, 253, 255, cv2.THRESH_BINARY)
    repaired = cv2.inpaint(denoised, m_art, 3, cv2.INPAINT_TELEA)
    gauss = cv2.GaussianBlur(repaired, (0, 0), 2.0)
    sharp = cv2.addWeighted(repaired, 1.6, gauss, -0.6, 0)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    return clahe.apply(sharp)

# --- HERO ---
st.markdown("""
<div class="hero">
    <div class="hero-eyebrow">▸ REVA UNIVERSITY — SCHOOL OF CSE — MINI PROJECT 2026</div>
    <div class="hero-title">
        A<span class="dot">.</span>R<span class="dot">.</span>E<span class="dot">.</span>S<span class="dot">.</span>
    </div>
    <div class="hero-fullname">ADVANCED · RADIOGRAPHIC · ENHANCEMENT · SYSTEM</div>
    <div class="hero-sub">PHASE_01 // 6_MODULE_PIPELINE // CHEST_XRAY_ENHANCEMENT // DEEP_LEARNING</div>
</div>
<div class="marine-line"></div>
""", unsafe_allow_html=True)

# --- PIPELINE ---
st.markdown("""
<div class="pipeline">
    <div class="pipe-step">01 · ORIENTATION</div>
    <div class="pipe-arrow">—</div>
    <div class="pipe-step">02 · SEGMENTATION</div>
    <div class="pipe-arrow">—</div>
    <div class="pipe-step">03 · DENOISE</div>
    <div class="pipe-arrow">—</div>
    <div class="pipe-step">04 · ARTIFACT_REMOVAL</div>
    <div class="pipe-arrow">—</div>
    <div class="pipe-step">05 · SUPER_RESOLUTION</div>
    <div class="pipe-arrow">—</div>
    <div class="pipe-step">06 · CONTRAST_ENHANCE</div>
</div>
<div class="marine-line"></div>
""", unsafe_allow_html=True)

# --- UPLOAD ---
st.markdown("<p class='upload-label'>▸ UPLOAD_RADIOGRAPHIC_INPUT // JPG / PNG / WebP / JPEG</p>", unsafe_allow_html=True)
up = st.file_uploader("", type=["jpg", "png", "jpeg","WebP"])

# --- PROCESS ---
if up:
    data = np.asarray(bytearray(up.read()), dtype=np.uint8)
    original = cv2.imdecode(data, cv2.IMREAD_GRAYSCALE)

    with st.spinner("EXECUTING A.R.E.S. PIPELINE..."):
        ai = load_engine()
        final = process_reva(original, ai)
        psnr_v = cv2.PSNR(original, final)
        ssim_v, _ = ssim(original, final, full=True)

    st.markdown("<div class='marine-line'></div>", unsafe_allow_html=True)

    # Images
    col1, col2 = st.columns(2, gap="medium")
    with col1:
        st.markdown("""<div class="img-card">
            <div class="img-label img-label-orig">
                <span class="img-label-dot dot-gray"></span>INPUT_BASELINE
            </div>
        </div>""", unsafe_allow_html=True)
        st.image(original, use_container_width=True)

    with col2:
        st.markdown("""<div class="img-card-enhanced">
            <div class="img-label img-label-enh">
                <span class="img-label-dot dot-marine"></span>ENHANCED_OUTPUT
            </div>
        </div>""", unsafe_allow_html=True)
        st.image(final, use_container_width=True)

    st.markdown("<div class='marine-line'></div>", unsafe_allow_html=True)

    # Quality labels
    quality  = "EXCELLENT" if psnr_v >= 35 else "NOMINAL" if psnr_v >= 28 else "ACCEPTABLE"
    psnr_tag = "↑ STRONG SIGNAL" if psnr_v >= 30 else "↑ NOMINAL"
    ssim_tag = "↑ HIGH FIDELITY" if ssim_v >= 0.85 else "↑ STRUCTURAL OK"

    # Metrics
    m1, m2, m3, m4 = st.columns(4)
    with m1:
        st.markdown(f"""<div class="metric-box">
            <div class="m-label">PSNR_SIGNAL</div>
            <div class="m-value">{psnr_v:.2f} dB</div>
            <div class="m-delta">{psnr_tag}</div>
        </div>""", unsafe_allow_html=True)
    with m2:
        st.markdown(f"""<div class="metric-box">
            <div class="m-label">SSIM_INDEX</div>
            <div class="m-value">{ssim_v:.4f}</div>
            <div class="m-delta">{ssim_tag}</div>
        </div>""", unsafe_allow_html=True)
    with m3:
        st.markdown(f"""<div class="metric-box">
            <div class="m-label">QUALITY_GRADE</div>
            <div class="m-value">{quality}</div>
            <div class="m-delta">6-MODULE PIPELINE</div>
        </div>""", unsafe_allow_html=True)
    with m4:
        st.markdown(f"""<div class="metric-box">
            <div class="m-label">PIPELINE_STATUS</div>
            <div class="m-value">DONE ✓</div>
            <div class="m-delta">ALL MODULES PASSED</div>
        </div>""", unsafe_allow_html=True)

    # Download
    st.markdown("<div style='height:24px'></div>", unsafe_allow_html=True)
    _, dl_col, _ = st.columns([2, 1, 2])
    with dl_col:
        buf = io.BytesIO()
        PILImage.fromarray(final).save(buf, format="PNG")
        st.download_button(
            label="⬇  DOWNLOAD_ENHANCED",
            data=buf.getvalue(),
            file_name="ARES_enhanced.png",
            mime="image/png",
            use_container_width=True,
        )

    # Status bar
    st.markdown(f"""
    <div class="status-bar">
        <span>A.R.E.S. // ADVANCED RADIOGRAPHIC ENHANCEMENT SYSTEM // REVA UNIVERSITY 2026</span>
        <span>PSNR_{psnr_v:.2f}dB // SSIM_{ssim_v:.4f} // QUALITY_{quality} // STATUS_COMPLETE</span>
    </div>
    """, unsafe_allow_html=True)

else:
    st.markdown("""
    <div class="empty-state">
        <div style="font-family:'JetBrains Mono',monospace;font-size:3.5rem;color:#0F2035;margin-bottom:20px;">⬡</div>
        <div style="font-family:'JetBrains Mono',monospace;font-size:0.65rem;color:#1A3A5A;letter-spacing:4px;">
            SYSTEM_READY // AWAITING_RADIOGRAPHIC_INPUT
        </div>
        <div style="font-family:'JetBrains Mono',monospace;font-size:0.58rem;color:#0F2035;letter-spacing:2px;margin-top:10px;">
            UPLOAD CHEST X-RAY TO INITIALIZE A.R.E.S. PIPELINE
        </div>
    </div>
    """, unsafe_allow_html=True)

Overwriting app.py


In [ ]:
# Install pyngrok if not already installed to ensure it's available
!pip install pyngrok -q

# Re-authenticate ngrok in case the previous cell didn't run or was reset
import os
os.system("ngrok config add-authtoken 3BIsHcBq4hrafFQwQUnMkda1zN8_41zk6LznS2U4CTcnQd4wT")

from pyngrok import ngrok
import subprocess, threading
import time # Import time for sleep

# Kill old tunnels (including any lingering processes)
ngrok.kill()
os.system('kill -9 $(pgrep ngrok)') # Forcefully kill any ngrok processes
time.sleep(2) # Give ngrok processes a moment to terminate

# Explicitly disconnect any existing tunnels from pyngrok's perspective
try:
    # This method seems to be deprecated or unavailable in this pyngrok version.
    # We rely on ngrok.kill() and os.system('kill -9...') for cleanup.
    pass # Removing the problematic line
except Exception as e:
    print(f"Error disconnecting tunnels: {e}")

# Start Streamlit
def run_st():
    subprocess.run(["python", "-m", "streamlit", "run", "app.py", "--server.port", "8501"])

threading.Thread(target=run_st).start()

# Create the public link
public_url = ngrok.connect(8501).public_url
print(f"\n--- CLICK THIS LINK FOR YOUR DEMO ---")
print(public_url)


--- CLICK THIS LINK FOR YOUR DEMO ---
https://nonextinguishable-nonrhetorically-teagan.ngrok-free.dev
